# 04c (native) - test predictions at the NATIVE resolution, both final-system paths: MODEL = "zs" (v1.5) or "ft" (v2) (thin wrapper)

GG decision 2026-10-03: the production resolution is native, `max_pixels` 2,196,480 = 2,145 visual
tokens per page. The pre-registered final-system rule (spec section 11) picks between ZS + rules and
FT + rules after the 3-fold out-of-fold comparison, so BOTH deliverable paths are ready here, behind
one switch. Qwen3.5-4B, IMAGE ONLY, KEYED output, prompt v2, greedy, xgrammar, seed 42, fp16,
logprobs on, the 200 test documents / 280 pages, OCR (the 04b cache) + R1 / R2 / R3 all ON. Test
images only: there are no test labels, nothing here scores anything, and no document value is
printed (counts, ids, hashes). The 1260-token notebooks 04 / 04b / 04c and their submissions are
untouched.

**`MODEL = "zs"` ('v1.5', default): the BASE model, no adapter.** The v0 traces are at 1260 tokens
and can NOT be reused (their config hash differs; the check cell proves the reuse decision refuses
them), so this path runs the VLM: smoke gate (5 dev documents), batch size = the one stored with the
02n run (`ZS_RUN_DIR`; the batch contract of 04: the test batch must EQUAL the zero-shot run's),
resumable run, determinism pass (5 seeded test documents decoded again in a fresh process, replaying
the exact generate calls of the first pass), then OCR, R1-R3 and the blocking checks of 04b (0 R2
skips, shapes sha256 = the repo file, every rule on). Output `submissions/v15_<sha7>/`.

**`MODEL = "ft"` ('v2'): the FINAL native adapter**, verified as in 04c (+ the resolution gate),
merged into the fp16 weights, guarded at the 02n batch size on the merged model, smoke-checked, run,
determinism pass, then the same checks as 04c. Its review flags need the zero-shot output of the same
documents at native: `ZS_TEST_DIR` = the validated `v15_<sha7>` folder of the zs path. No zero-shot
inference ever runs on this path. Output `submissions/v2n_<sha7>/`.

Flow: parameters -> Drive + secrets -> clone at the pin -> unzip data incl. test images (200 docs /
280 pages by file name) -> install (`uv sync --frozen --group vlm --group train`, no Paddle) ->
**NATIVE CHECKS (CPU, before anything runs: refuses, never advises)** -> ESTIMATE (T4 hours / CU,
UNVERIFIED) -> plan -> OCR (adopt the 04b record when `REUSE_OCR_FROM` is set; the OCR stage checks
the cached pages: resumable, 280/280 asserted, 5-page seeded re-OCR) -> [zs: smoke, batch, run,
determinism] / [ft: **resolution gate + adapter verification**, infer (merge, guard, smoke, run),
determinism] -> assemble with R1-R3 + validation (`predict_native finalize`: the 04b / 04c checks +
the native checks) -> **REVIEW FLAGS** -> banner VALIDATED / REJECTED.

What the native checks refuse (`python -m shipdoc.predict_native check`, then again by stage):
a config whose `max_pixels` is not 2,196,480 (or not the native name); a 02n run that is missing,
incomplete or whose manifest config hash is not the native config's (a 1260-token zero-shot run is
refused); no stored batch size (and a `BATCH_SIZE` different from the 02n run's on the zs path);
`ft`: an adapter trained at another resolution (`shipdoc.resmatch`, fail closed), a missing adapter
manifest, a `ZS_TEST_DIR` that is missing, not VALIDATED, not at the native config hash, not a
`v15_` full-mode non-fine-tuned run, without `test_predictions.json`, with a batch size other than
the 02n run's, or refused by the strict reuse decision; `zs`: a `REUSE_V0_DIR` whose traces the reuse
decision would ALLOW (it must refuse them on the config hash). The `ft` stages repeat the resolution
gate before `predict_ft` runs and stamp it into `ft_run.json`; `finalize` rejects an `ft` run without
that stamp. The flags refuse (nothing is computed): no `CALIBRATOR_FILE` (the default: it says to
refreeze at native with `scripts/freeze_calibrator.py --arm <zs|ft>`), a calibrator of the other arm,
a calibrator that does not carry the native config hash (`run_config_hash`; the 1260-token
`meta/calibrator_zs.json` carries none and is refused), and a submission whose manifest is not at the
native config hash.

Parameters (every required input is checked and refuses with a clear message): `PINNED_SHA`; `MODEL`;
`ZS_SHA7` (the sha7 of the 02n pin; the zero-shot folder is
`zeroshot500_qwen35_4b_img_only_native_<ZS_SHA7>`; the placeholder refuses); `BATCH_SIZE` (None = the
02n run's stored size, refused if absent); `ft` only: `TRAIN_SHA7` / `PRECISION` / `ADAPTER_DIR` (the
03n `final` folder, default `runs/ft_native_final_<TRAIN_SHA7>_<PRECISION>/final`; the placeholder
refuses) and `ZS_TEST_DIR` (default `submissions/v15_<PINNED_SHA[:7]>`, i.e. the zs run of the SAME
pin); `REUSE_V0_DIR` (zs: the v0 folder the reuse decision must refuse; None skips); `REUSE_OCR_FROM`
(default the 04b folder `submissions/v1_8833c73`: its OCR timing and re-check record are adopted, the
cache itself is the one on Drive); `REUSE_ACK_SPIKE_DIFF`; `CALIBRATOR_FILE` (None: the flags cell
refuses; relative paths are looked up in the clone, then on Drive); `FIELD_TARGET` / `DOC_TARGET`.

Merge precision (ft): the adapter's low-rank update is added to the fp16 base weights (fp32 sum
rounded to fp16); the merged model is not bit-identical to base + unmerged adapter. The whole of both
paths at native (batch size, VRAM, the merge, every timing) is UNVERIFIED on a GPU until run.

Output: `MyDrive/shipdoc-extract/submissions/v15_<sha7>/` (zs) or `v2n_<sha7>/` (ft) with
`test_predictions.json`, `review_flags.json`, `trace.jsonl`, `manifest.json`,
`validation_report.json`, `rules.jsonl`, `ocr_timing.json`; on a failed check the predictions are
`test_predictions.REJECTED.json`. Download by hand into `$SHIPDOC_SUBMISSIONS_DIR\<folder>\`. If the
session dies, Run all: every stage resumes. See notebooks/README.md.


In [ ]:
# Parameters. PINNED_SHA = the repo commit this run is pinned to (full 40-char SHA).
PINNED_SHA = "3e88cc8aa4bc2c33ac676c4c52395271f570b9cd"
MODEL = "zs"  # "zs" = v1.5, the base model, no adapter | "ft" = v2, the final native adapter
RUN_NAME = "testnative"
CONFIG = "qwen35_4b_img_only_native"  # KEYED, prompt v2, max_pixels 2196480: production
SPLIT = "test"
EXPECTED_DOCS = 200
EXPECTED_PAGES = 280
SMOKE_DOCS = "splits/smoke5.json"  # 5 dev docs picked by meta tags (shipdoc.smoke)
BENCH_DOCS = "splits/bench12.json"  # 12 dev pages for the batch guard / bench (shipdoc.bench)
SHARD = "0/1"  # fixed: one tab (the plan stage and the assemble step take it)
# sha7 of the 02n pin: the 02n run folder is zeroshot500_qwen35_4b_img_only_native_<ZS_SHA7>.
ZS_SHA7 = PINNED_SHA[:7]  # public build: one pin for all native notebooks
ZS_RUN_DIR = f"zeroshot500_qwen35_4b_img_only_native_{ZS_SHA7}"  # the 02n run folder under runs/
BATCH_SIZE = None  # None = the batch size stored with the 02n run (refused if absent); or an int
# ft only: the NATIVE 03n final run (its folder is printed by the 03n banner) and the zs submission
TRAIN_SHA7 = PINNED_SHA[:7]  # public build: one pin for all native notebooks
PRECISION = "bf16"  # bf16 (L4) | fp16 (T4): the suffix of the 03n run folder
# the final run's final/ folder: relative to MyDrive/shipdoc-extract, or an absolute path
ADAPTER_DIR = f"runs/ft_native_final_{TRAIN_SHA7}_{PRECISION}/final"
ZS_TEST_DIR = f"submissions/v15_{PINNED_SHA[:7]}"  # ft only: the VALIDATED zs run (same pin)
REUSE_V0_DIR = None  # public build: no 1260-token folder
REUSE_ACK_SPIKE_DIFF = False  # True accepts a spike.py diff beyond the allowlisted opt-in ones
CALIBRATOR_FILE = None  # None = no flags (refuses); else the native calibrator, see the README
FIELD_TARGET = 0.98  # precision target of the per-field-type tau (0.95 | 0.98 | 0.99)
DOC_TARGET = 0.98  # precision target of the document auto-accept tau (0.95 | 0.98)
REUSE_OCR_FROM = None  # public build: OCR runs here

if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell (the pin commit fills it).")
if MODEL not in ("zs", "ft"):
    raise ValueError(f"MODEL must be 'zs' or 'ft', got {MODEL!r}")
if not (isinstance(ZS_SHA7, str) and len(ZS_SHA7) == 7
        and all(c in "0123456789abcdef" for c in ZS_SHA7)):
    raise ValueError(f"ZS_SHA7 must be the 7 hex characters of the 02n pin, got {ZS_SHA7!r}")
if BATCH_SIZE is not None and (isinstance(BATCH_SIZE, bool) or not isinstance(BATCH_SIZE, int)
                               or BATCH_SIZE < 1):
    raise ValueError(f"BATCH_SIZE must be None or an int >= 1, got {BATCH_SIZE!r}")
if PRECISION not in ("bf16", "fp16"):
    raise ValueError(f"PRECISION must be 'bf16' or 'fp16', got {PRECISION!r}")
if MODEL == "ft":
    if not (isinstance(TRAIN_SHA7, str) and len(TRAIN_SHA7) == 7
            and all(c in "0123456789abcdef" for c in TRAIN_SHA7)):
        raise ValueError(f"TRAIN_SHA7 must be 7 hex characters (03n pin), got {TRAIN_SHA7!r}")
    if not isinstance(ADAPTER_DIR, str) or not ADAPTER_DIR or "FILL" in ADAPTER_DIR:
        raise ValueError("ADAPTER_DIR must name the native final run's final/ folder on Drive.")
    if not isinstance(ZS_TEST_DIR, str) or not ZS_TEST_DIR or "FILL" in ZS_TEST_DIR:
        raise ValueError("ZS_TEST_DIR must be the validated zs submission folder (v15_<sha7>).")
if REUSE_V0_DIR is not None and not isinstance(REUSE_V0_DIR, str):
    raise ValueError(f"REUSE_V0_DIR must be None or a path string, got {REUSE_V0_DIR!r}")
if not isinstance(REUSE_ACK_SPIKE_DIFF, bool):
    raise ValueError(f"REUSE_ACK_SPIKE_DIFF must be True or False, got {REUSE_ACK_SPIKE_DIFF!r}")
if CALIBRATOR_FILE is not None and not (isinstance(CALIBRATOR_FILE, str) and CALIBRATOR_FILE):
    raise ValueError(f"CALIBRATOR_FILE must be None or a path string, got {CALIBRATOR_FILE!r}")
if FIELD_TARGET not in (0.95, 0.98, 0.99):
    raise ValueError(f"FIELD_TARGET must be 0.95, 0.98 or 0.99, got {FIELD_TARGET!r}")
if DOC_TARGET not in (0.95, 0.98):
    raise ValueError(f"DOC_TARGET must be 0.95 or 0.98, got {DOC_TARGET!r}")
if REUSE_OCR_FROM is not None and not isinstance(REUSE_OCR_FROM, str):
    raise ValueError(f"REUSE_OCR_FROM must be None or a path string, got {REUSE_OCR_FROM!r}")
SHA7 = PINNED_SHA[:7]
SUB_PREFIX = "v15" if MODEL == "zs" else "v2n"  # the submission folder of this path
SUB_NAME = f"{SUB_PREFIX}_{SHA7}"
RUN_ID = f"{RUN_NAME}_{MODEL}_{SHA7}"  # MyDrive/shipdoc-extract/runs/<RUN_ID>/; never a 04* name
CFG = f"configs/spike_{CONFIG}.yaml"
MODE = "run"  # the shared 04 cells take these (one tab, no shards)
SHARD_I, SHARD_K = 0, 1
SHARD_RUN_ID = RUN_ID
TAB = "0of1"


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
REQUIRED_ZIPS = ["data.zip", "assignment.zip"]
RUNS_DIR = DRIVE_DIR / "runs"

drive.mount("/content/drive")

print("Public build: any Google account; inputs are read from", DRIVE_DIR)

missing = [n for n in REQUIRED_ZIPS if not (DRIVE_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in MyDrive/shipdoc-extract/: {', '.join(missing)}")
print("Drive inputs present:", ", ".join(REQUIRED_ZIPS))
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR = DRIVE_DIR / "submissions"  # test predictions live here, not in the repo
SUBMISSION_DIR = SUBMISSIONS_DIR / SUB_NAME
META_DIR = RUNS_DIR / f"{RUN_ID}_meta"  # plan, smoke status, batch decision, OCR records, check
RUN_DIR = RUNS_DIR / RUN_ID
OCR_ROOT = DRIVE_DIR / "ocr_cache_test"  # <root>/paddleocr/test/<page>.json, shared with 04b
for _d in (SUBMISSIONS_DIR, META_DIR, SUBMISSION_DIR, OCR_ROOT):
    _d.mkdir(parents=True, exist_ok=True)
PLAN_PATH = META_DIR / "plan.json"
TEST_DOCS_PATH = META_DIR / "test_docs.json"  # ids only
SMOKE_STATUS_PATH = META_DIR / "smoke_status.json"
DECISION_PATH = META_DIR / "batch_decision.json"
CHECK_PATH = META_DIR / "native_check.json"


def _under_runs(p: str) -> Path:
    """Absolute as given; 'runs/...' under MyDrive/shipdoc-extract; a bare name under runs/."""
    if Path(p).is_absolute():
        return Path(p)
    return DRIVE_DIR / p if p.startswith("runs/") else RUNS_DIR / p


def _under_drive(p: str) -> Path:
    """Absolute as given; else relative to MyDrive/shipdoc-extract."""
    return Path(p) if Path(p).is_absolute() else DRIVE_DIR / p


ZS_RUN = _under_runs(ZS_RUN_DIR)  # the 02n run: batch size, config hash, the batch contract
DEV_RUN = ZS_RUN  # shipdoc predict assemble --dev-run-dir: the test batch must equal the 02n run's
ADAPTER = _under_drive(ADAPTER_DIR) if MODEL == "ft" else None
ZS_TEST = _under_drive(ZS_TEST_DIR) if MODEL == "ft" else None
V0_DIR = _under_drive(REUSE_V0_DIR) if (MODEL == "zs" and REUSE_V0_DIR) else None
OCR_FROM = _under_drive(REUSE_OCR_FROM) if REUSE_OCR_FROM else None
_required = [("zero-shot run (02n)", ZS_RUN, "manifest.json"),
             ("zero-shot run (02n)", ZS_RUN, "predictions.json")]
if MODEL == "ft":
    _required += [("final adapter folder", ADAPTER, "manifest.json"),
                  ("zs test submission (ZS_TEST_DIR)", ZS_TEST, "manifest.json")]
for label, path, need in _required:
    if not (path / need).is_file():
        raise FileNotFoundError(
            f"{label} {path} has no {need}. MODEL={MODEL!r} needs: ZS_SHA7 (the 02n pin)"
            + (", TRAIN_SHA7 / PRECISION / ADAPTER_DIR (the 03n final run) and ZS_TEST_DIR (the "
               "validated v15_<sha7> folder: run this notebook with MODEL = 'zs' first)"
               if MODEL == "ft" else "") + " in the parameters cell."
        )
print("run_id:", RUN_ID, "| model:", MODEL, "| batch size:", BATCH_SIZE or "from the 02n run")
print("zero-shot run (02n):", ZS_RUN)
print("final adapter      :", ADAPTER, "| zs test folder:", ZS_TEST)
print("OCR cache folder   :", OCR_ROOT, "| adopt OCR record from:", OCR_FROM)
print("Drive run folder   :", RUN_DIR)
print("Drive submission   :", SUBMISSION_DIR)


In [ ]:
import os

from google.colab import userdata


def get_secret(name: str, required: bool) -> str | None:
    """Read a Colab secret without ever printing it."""
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError / NotebookAccessError: both mean "not usable"
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret {name} is missing or notebook access is off "
            "(left sidebar -> key icon -> add it and enable Notebook access)."
        )
    return value or None


HF_TOKEN = get_secret("HF_TOKEN", required=False)
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (optional)")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gaurav-gandhi-2411/shipdoc-extract-public.git"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
import json
import re
import shutil
import zipfile

LOCAL_ZIPS = Path("/content/zips")
LOCAL_ZIPS.mkdir(exist_ok=True)
CONTENT = Path("/content")
# data.zip holds data/ and assignment.zip holds assignment/; both unzip to /content (not into the
# clone), and the CLI is pointed at them with SHIPDOC_DATA_DIR / SHIPDOC_ASSIGNMENT_DIR below.
for name in ("data.zip", "assignment.zip"):
    local = LOCAL_ZIPS / name
    if not local.is_file() or local.stat().st_size != (DRIVE_DIR / name).stat().st_size:
        shutil.copyfile(DRIVE_DIR / name, local)  # local disk: Drive reads are slow
    with zipfile.ZipFile(local) as zf:
        files = [n for n in zf.namelist() if not n.endswith("/")]
        if any(not (CONTENT / n).is_file() for n in files):  # also repairs a half-extracted tree
            zf.extractall(CONTENT)
    print(f"{name}: {len(files)} files under {CONTENT / files[0].split('/')[0]}")

DATA_DIR, ASSIGNMENT_DIR = CONTENT / "data", CONTENT / "assignment"
DEV_LABELS = DATA_DIR / "dev" / "labels"  # the smoke gate and the bench use DEV docs only
TEST_IMAGES = DATA_DIR / SPLIT / "images"
SCHEMA_PATH = ASSIGNMENT_DIR / "schema.json"
SAMPLE_PATH = ASSIGNMENT_DIR / "sample_submission.json"
assert SCHEMA_PATH.is_file(), "assignment.zip has no schema.json"
assert TEST_IMAGES.is_dir(), f"data.zip has no {TEST_IMAGES}: the test images are required"
# There are no test labels and this notebook never looks for any.
assert not (DATA_DIR / SPLIT / "labels").exists(), "unexpected test labels in data.zip: stop"
STEM = re.compile(r"^(.+)_p(\d+)$")
_pages = {}
for _p in TEST_IMAGES.iterdir():
    _m = STEM.match(_p.stem)
    if _p.is_file() and _m:
        _pages[_m.group(1)] = _pages.get(_m.group(1), 0) + 1  # ids from FILE NAMES only
test_pages_by_doc = dict(sorted(_pages.items()))
assert len(test_pages_by_doc) == EXPECTED_DOCS, f"{len(test_pages_by_doc)} test docs found"
assert sum(test_pages_by_doc.values()) == EXPECTED_PAGES, "test page count differs"
print(f"data OK: {EXPECTED_DOCS} test docs / {EXPECTED_PAGES} page images (no labels); dev labels:",
      DEV_LABELS.is_dir())


# Public build: the evaluators' package (assignment.zip, never vendored in the repository) must be
# complete BEFORE the install and model cells; the post-processing rules read its schema.json.
for _need in ("schema.json", "score.py"):
    assert (ASSIGNMENT_DIR / _need).is_file(), (
        f"assignment.zip has no assignment/{_need}: upload BOTH data.zip and assignment.zip "
        "(your evaluator package) to MyDrive/shipdoc-extract/"
    )


In [ ]:
# ESTIMATE (UNVERIFIED) of the OCR stage: 280 test pages, PaddleOCR PP-OCRv5
# server
# with the document + text-line orientation classifiers (shipdoc.ocr). No GPU measurement of
# it exists in this repo: the GPU figure is an ASSUMPTION. The CPU figure is the only
# measurement: 10.16 s/page mean over 951 pages (reports/ocr.md, local 8-thread CPU, shared
# with other jobs); a free Colab
# runtime has fewer cores, so the Colab CPU figure below doubles it (also an ASSUMPTION).
import json

OCR_GPU_S_PER_PAGE = 2.0  # ASSUMED, not measured
OCR_CPU_S_PER_PAGE = 2 * 10.16  # local measurement x 2 (ASSUMED Colab CPU slowdown)
OCR_SETUP_MIN = 10.0  # ASSUMED: venv build + paddle wheel + model download + first model load
_speed = json.loads((REPO / "configs" / "spike_speed.json").read_text(encoding="utf-8"))
_rate = _speed.get("t4_cu_per_hour")
_gpu_h = (OCR_SETUP_MIN * 60 + EXPECTED_PAGES * OCR_GPU_S_PER_PAGE) / 3600
_cpu_h = (OCR_SETUP_MIN * 60 + EXPECTED_PAGES * OCR_CPU_S_PER_PAGE) / 3600
print("=" * 100)
print("ESTIMATE (UNVERIFIED) OCR stage, resumable per page (pages already cached are skipped):")
print(f"  GPU (T4): {_gpu_h:.2f} h incl. {OCR_SETUP_MIN:.0f} min setup "
      f"({OCR_GPU_S_PER_PAGE} s/page ASSUMED), ~{_gpu_h * _rate:.2f} CU @ {_rate} CU/h")
print(f"  CPU fallback: {_cpu_h:.2f} h ({OCR_CPU_S_PER_PAGE:.1f} s/page ASSUMED), CU rate of a "
      "CPU runtime is not in configs/spike_speed.json: n/a")
print("  + a seeded 5-page re-OCR in a fresh process (about one model load + 5 pages).")
print(f"REUSE_OCR_FROM = {REUSE_OCR_FROM!r}: pages already cached on Drive cost nothing (the OCR "
      "stage checks the cache first; a complete, valid cache is only checked, not re-run).")
print("=" * 100)


In [ ]:
import collections
import subprocess

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(DATA_DIR),
    "SHIPDOC_ASSIGNMENT_DIR": str(ASSIGNMENT_DIR),
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    "SHIPDOC_OCR_CACHE": str(OCR_ROOT),  # the test OCR cache that R2 reads
    "SHIPDOC_SUBMISSIONS_DIR": str(SUBMISSIONS_DIR),
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
}
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
VLM_GROUP = ["--group", "vlm", "--group", "train"]  # peft==0.21.2 comes from the locked train group
rc, _ = run_stream(["uv", "sync", "--frozen", *VLM_GROUP, "--python", "3.11"])
assert rc == 0, f"uv sync failed (exit {rc})"

subprocess.run(["nvidia-smi"], check=False)
PROBE = (
    "import json, importlib.metadata as m, torch;"
    "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
    " 'arch': torch.cuda.get_arch_list(), 'name': torch.cuda.get_device_name(0),"
    " **{p: m.version(p) for p in ('transformers', 'xgrammar', 'peft', 'pycountry')}}))"
)
info = json.loads(subprocess.run([PY, "-c", PROBE], env=ENV, capture_output=True, text=True,
                                 check=True).stdout.strip().splitlines()[-1])
print(info)
assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> T4 GPU."
# T4 is sm_75: need a native sm_75 kernel or PTX (compute_XX with XX <= 75) it can JIT.
ok = "sm_75" in info["arch"] or any(
    a.startswith("compute_") and int(a.split("_")[1]) <= 75 for a in info["arch"]
)
assert ok, f"torch build has no sm_75 support: {info['arch']}"

print(f"GPU {info['name']}: written for a T4 (fp16, sm_75); other GPUs also run it")
_no_ocr = (
    "import importlib.util as u, sys; sys.exit(0 if u.find_spec('paddle') is None else 1)"
)
assert subprocess.run([PY, "-c", _no_ocr], env=ENV, check=False).returncode == 0, (
    "the OCR stack is in the decoding venv: Paddle runs ONLY in /content/ocr-venv"
)


In [ ]:
# NATIVE CHECKS (CPU, nothing runs on the GPU before they pass). `shipdoc.predict_native check`
# REFUSES, with the reason, unless: the config is the native one; the 02n run is complete and
# carries the native config hash; its stored batch size resolves (the zs test batch must EQUAL it);
# ft: the adapter passes the resolution gate (shipdoc.resmatch, fail closed) and ZS_TEST_DIR is a
# VALIDATED v15 folder at the native hash that the strict reuse decision accepts; zs: the v0 folder
# (when present) is REFUSED for reuse on the config hash. It writes the resolved batch size.
check_cmd = [PY, "-m", "shipdoc.predict_native", "check", "--model", MODEL, "--config", CFG,
             "--zs-run-dir", str(ZS_RUN), "--out", str(CHECK_PATH)]
if BATCH_SIZE is not None:
    check_cmd += ["--batch-size", str(BATCH_SIZE)]
    print("=" * 78)
    print(f"MANUAL BATCH SIZE {BATCH_SIZE}: the zs path REFUSES a size other than the 02n run's;")
    print("the ft path lets the guard on the merged model decide (it may fall back to 1).")
    print("=" * 78)
if MODEL == "ft":
    check_cmd += ["--adapter-dir", str(ADAPTER), "--zs-test-dir", str(ZS_TEST)]
elif V0_DIR is not None and V0_DIR.is_dir():
    check_cmd += ["--v0-dir", str(V0_DIR)]
else:
    print("zs: no v0 folder to prove the reuse refusal on (REUSE_V0_DIR); the VLM runs anyway.")
if REUSE_ACK_SPIKE_DIFF:
    check_cmd.append("--ack-spike-diff")
    print("REUSE_ACK_SPIKE_DIFF = True: a spike.py diff beyond the allowlist is accepted.")
rc, tail = run_stream(check_cmd)
if rc != 0:
    raise RuntimeError(
        f"NATIVE CHECK REFUSED (exit {rc}): {tail[-3:]}. Nothing was run; read the refusal above "
        "(counts, hashes and names only), fix the parameter it names and Run all again."
    )
check = json.loads(CHECK_PATH.read_text())
BATCH = int(check["batch_size"])  # the 02n run's stored size (or the ft override)
FLAGS_ZS_BATCH = int(check.get("zs_test_batch_size") or BATCH)  # what the v15 run was decoded at
print(f"NATIVE CHECKS PASSED: model {MODEL}, batch size {BATCH} ({check['batch_source']}), "
      f"config hash {check['config_hash']}")


In [ ]:
# ESTIMATE (UNVERIFIED) of the GPU session of this path: T4 hours and compute units, low / high
# pace. Nothing in it is measured at native: low = the 02 batch-8 pace (MEASURED at 1260 tokens)
# x the 06 sweep's native cost ratio, valid only if the batch size fits; high = the sweep's
# MEASURED native batch-1 pace (shipdoc.nativerun). ASSUMED 120 s model load, 60 s merge. No
# confirmation prompt (Run all must not block): interrupt the run yourself if it is too high.
est_cmd = [PY, "-m", "shipdoc.predict_native", "estimate", "--model", MODEL, "--config", CFG,
           "--zs-run-dir", str(ZS_RUN), "--smoke-docs", SMOKE_DOCS]
if BATCH_SIZE is not None:
    est_cmd += ["--batch-size", str(BATCH_SIZE)]
rc, tail = run_stream(est_cmd)
assert rc == 0, f"estimate failed (exit {rc}): {tail[-3:]}"
print("THIS RUN: model", MODEL, "| batch size", BATCH, "| OCR record adopted from:", REUSE_OCR_FROM)


In [ ]:
# Plan: document ids come from the test image FOLDER NAMES (never labels); the shard's
# documents and the 5 seeded determinism documents are fixed here (rule in the output).
rc, tail = run_stream([PY, "-m", "shipdoc", "predict", "plan", "--shard", SHARD,
                       "--out", str(PLAN_PATH)])
assert rc == 0, f"plan failed (exit {rc}): {tail[-3:]}"
plan = json.loads(PLAN_PATH.read_text())
assert plan["n_docs"] == EXPECTED_DOCS and plan["n_pages"] == EXPECTED_PAGES, "plan counts differ"
assert sorted(plan["docs"]) == sorted(test_pages_by_doc), "plan ids differ from the file names"
TEST_DOCS_PATH.write_text(json.dumps(plan["docs"]))
print("determinism rule:", plan["determinism_rule"])


In [ ]:
# Optional: adopt the OCR timing and the OCR re-check record of a finished 04b folder (the OCR cache
# on Drive is shared, so the pages are already there). Only the records are copied, never over an
# existing file; the OCR stage below still asserts 280/280 valid cached pages. A refusal is not
# fatal: the OCR stage then does its own timing and re-check.
if OCR_FROM is None:
    print("REUSE_OCR_FROM is None: the OCR stage below times and re-checks the cache itself.")
else:
    adopt_cmd = [PY, "-m", "shipdoc.predict_ft", "adopt-ocr", "--from-dir", str(OCR_FROM),
                 "--timing-out", str(SUBMISSION_DIR / "ocr_timing.json"),
                 "--recheck-out", str(META_DIR / "ocr_recheck.json"),
                 "--expect-pages", str(EXPECTED_PAGES)]
    rc, tail = run_stream(adopt_cmd)
    if rc != 0:
        print("!" * 78)
        print("OCR RECORD NOT ADOPTED (exit", rc, "): the OCR stage below runs its own checks.")
        print("!" * 78)


In [ ]:
# OCR of the 280 TEST pages, in its OWN venv (paddlepaddle-gpu clashes with the vlm
# group, so Paddle is never installed into or imported by the environment that decodes: this
# cell only starts subprocesses). Resumable: pages already cached AND valid are skipped (an
# invalid page file is renamed to *.invalid, never deleted, and redone). GPU when the runtime
# has one and the GPU wheel installed, else CPU with a LOUD warning. OCR text is never
# printed: counts and seconds only.
# Asserts 280/280 valid cached pages, writes ocr_timing.json (pages, seconds total, seconds/page
# mean and p95, device) to the submission folder, then re-OCRs 5 seeded pages in a fresh process.
import json
import subprocess
import time

OCR_VENV = Path("/content/ocr-venv")  # never the venv that decodes
assert OCR_VENV.resolve() != (REPO / ".venv").resolve(), "OCR must not share the decoding venv"
OCR_SH = ["bash", str(REPO / "scripts" / "colab_ocr_test.sh")]
OCR_ENV = {**ENV, "OCR_VENV": str(OCR_VENV)}
OCR_TIMING_PATH = SUBMISSION_DIR / "ocr_timing.json"
OCR_RECHECK_PATH = META_DIR / "ocr_recheck.json"
STAGE = [PY, "-m", "shipdoc.ocr_stage"]
PAGE_ARGS = ["--images-dir", str(TEST_IMAGES), "--cache-root", str(OCR_ROOT),
             "--expect-pages", str(EXPECTED_PAGES)]


def has_gpu() -> bool:
    try:
        res = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30,
                             check=False)
    except (OSError, subprocess.TimeoutExpired):
        return False
    return res.returncode == 0 and "GPU" in res.stdout


def ocr_device() -> dict:
    """Device from the runtime and the OCR venv's paddle flavour; loud on a CPU fallback."""
    flavor_file = OCR_VENV / ".paddle_flavor"
    flavor = flavor_file.read_text().strip() if flavor_file.is_file() else ""
    res = subprocess.run([*STAGE, "device", "--has-gpu", "1" if has_gpu() else "0",
                          "--flavor", flavor], cwd=REPO, env=ENV, capture_output=True, text=True,
                         check=False)
    info = json.loads(res.stdout.strip().splitlines()[-1])
    for warning in info["warnings"]:
        print("!" * 78)
        print("OCR DEVICE WARNING:", warning)
        print("!" * 78)
    if not info["ok"]:
        raise RuntimeError(f"OCR cannot run on this runtime: {info['warnings']}")
    print(f"OCR device: {info['device']} (paddle flavour {info['flavor'] or 'unknown'})")
    return info


def ocr_status(tag: str) -> dict:
    path = META_DIR / f"ocr_check_{tag}.json"
    rc, _ = run_stream([*STAGE, "check", *PAGE_ARGS, "--isolate-invalid", "--out", str(path)])
    return {"rc": rc, **json.loads(path.read_text())}


before = ocr_status("before")
wall, device = None, None
if before["rc"] == 0:
    print(f"OCR cache already complete ({before['valid']}/{EXPECTED_PAGES}): no OCR run")
else:
    print(f"OCR cache: {before['valid']}/{EXPECTED_PAGES} valid; running the missing pages")
    t_ocr = time.time()
    rc, tail = run_stream([*OCR_SH, "setup"], env=OCR_ENV)
    assert rc == 0, f"OCR venv setup failed (exit {rc}): {tail[-3:]}"
    device = ocr_device()["device"]
    rc, tail = run_stream([*OCR_SH, "run", str(DATA_DIR), str(OCR_ROOT)], env=OCR_ENV)
    wall = time.time() - t_ocr
    print(f"--- OCR run exited {rc} after {wall / 60:.1f} min (the check below decides) ---")
after = ocr_status("after")
if after["rc"] != 0:
    raise RuntimeError(
        f"OCR INCOMPLETE: {after['valid']}/{EXPECTED_PAGES} valid pages, {after['missing']} "
        f"missing {after['missing_pages']}, {after['invalid']} invalid. Everything cached is on "
        "Drive: Run all again resumes the rest."
    )
assert after["valid"] == EXPECTED_PAGES, f"{after['valid']} cached OCR pages, not {EXPECTED_PAGES}"
prior = json.loads(OCR_TIMING_PATH.read_text()) if OCR_TIMING_PATH.is_file() else {}
device = device or prior.get("device") or "unknown (the cache was complete at the start)"
fin = [*STAGE, "finalize", *PAGE_ARGS, "--device", device, "--out", str(OCR_TIMING_PATH)]
if wall is not None:
    fin += ["--wall-s", f"{wall:.1f}", "--new-pages", str(after["valid"] - before["valid"])]
rc, tail = run_stream(fin)
assert rc == 0, f"ocr timing failed (exit {rc}): {tail[-3:]}"
assert subprocess.run([PY, "-c", "import importlib.util as u, sys; "
                       "sys.exit(0 if u.find_spec('paddle') is None else 1)"],
                      env=ENV, check=False).returncode == 0, "Paddle leaked into the decoding venv"

# OCR determinism: 5 seeded pages (random.Random(42)) OCR'd AGAIN in a fresh process (the OCR
# venv, into a scratch cache); blocking criterion = identical page text (what R2 reads), the whole
# page JSON minus `seconds` is reported too. A failure is recorded and REJECTS the submission.
recheck = json.loads(OCR_RECHECK_PATH.read_text()) if OCR_RECHECK_PATH.is_file() else {}
if recheck.get("ok") is True:
    print("OCR recheck already passed:", recheck["n_text_identical"], "/", recheck["n_pages"])
else:
    rc, tail = run_stream([*OCR_SH, "setup"], env=OCR_ENV)  # a no-op unless the VM is new
    assert rc == 0, f"OCR venv setup failed (exit {rc}): {tail[-3:]}"
    ocr_device()
    rc, tail = run_stream([*OCR_SH, "module", "recheck", *PAGE_ARGS, "--scratch-root",
                           "/content/ocr_recheck", "--out", str(OCR_RECHECK_PATH)], env=OCR_ENV)
    if rc != 0:
        print("!" * 78)
        print("OCR RE-CHECK DID NOT PASS (exit", rc, "): the submission will be REJECTED. Report")
        print("back with", OCR_RECHECK_PATH, "(page names and booleans only).")
        print("!" * 78)


In [ ]:
if MODEL == "zs":
    # MANDATORY smoke gate (the same run and 7 checks as notebook 02, 5 DEV docs) at the native
    # config: the status file is bound to this code, config and model revision; the test run
    # refuses to start unless it says `passed`. A failure raises.
    SMOKE_RUN_ID = f"smoke_{SHARD_RUN_ID}"
    CFG = f"configs/spike_{CONFIG}.yaml"
    smoke_cmd = [PY, "-m", "shipdoc", "predict", "smoke", "--config", CFG,
                 "--docs", SMOKE_DOCS, "--run-id", SMOKE_RUN_ID, "--status", str(SMOKE_STATUS_PATH)]
    rc, tail = run_stream(smoke_cmd)
    SMOKE_PASSED = rc == 0
    if not SMOKE_PASSED:
        raise RuntimeError(
            "SMOKE GATE FAILED: the test run is NOT started. GG: report back with "
            f"{SMOKE_STATUS_PATH} and the per-assertion table printed above (dev documents only). "
            "Do not edit thresholds to make it pass."
        )
    print("SMOKE GATE PASSED: continuing.")

else:
    print("skipped (smoke gate): MODEL == 'zs' is false")


In [ ]:
if MODEL == "zs":
    # BATCH SIZE = the one stored with the 02n run (the bench there measured it at native). MANUAL
    # on purpose: `shipdoc predict batch --batch-size` skips a second bench (a stored bench result
    # applies only to its own code SHA) and still enforces the batch-size contract against the 02n
    # run (`--dev-run-dir`): greedy outputs only compare at equal batch size. A mismatch raises.
    if not SMOKE_PASSED:
        raise RuntimeError("The smoke gate has not passed: batch size and test run are NOT run.")
    BENCH_DIR = RUNS_DIR / f"bench_{CONFIG}_{SHA7}"
    batch_cmd = [PY, "-m", "shipdoc", "predict", "batch", "--config", CFG,
                 "--bench-docs", BENCH_DOCS, "--bench-dir", str(BENCH_DIR),
                 "--decision", str(DECISION_PATH), "--dev-run-dir", str(DEV_RUN),
                 "--batch-size", str(BATCH)]
    rc, tail = run_stream(batch_cmd)
    if rc != 0:
        raise RuntimeError(f"BATCH SIZE REFUSED (exit {rc}): {tail[-3:]}. Test run NOT started.")
    decision = json.loads(DECISION_PATH.read_text())
    assert int(decision["batch_size"]) == BATCH, "the decision's batch size is not the 02n run's"
    print(f"BATCH SIZE FOR THE TEST RUN: {BATCH} ({decision['source']}); dev contract: "
          f"{decision['contract']}")

else:
    print("skipped (batch size): MODEL == 'zs' is false")


In [ ]:
if MODEL == "zs":
    # Test run over the 200 documents, one subprocess, RESUMABLE per doc (as 04): the CLI
    # appends one line to trace.jsonl and rewrites predictions.json / progress.json after every
    # document; a resume skips the doc_ids already traced. Refuses unless the smoke gate passed.
    import re
    import threading
    import time
    from datetime import UTC, datetime

    if not SMOKE_PASSED or "BATCH" not in globals():
        raise RuntimeError("Smoke gate or batch decision missing: the test run is NOT started.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)
    SESSIONS_PATH = RUN_DIR / "sessions.json"  # wall-clock log, one entry per Run all
    sessions = json.loads(SESSIONS_PATH.read_text()) if SESSIONS_PATH.is_file() else []
    THIS_DOCS = plan["shard_n_docs"]


    def save_json(path, obj) -> None:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(obj, indent=1))
        tmp.replace(path)  # atomic: a disconnect never leaves a half-written file


    def read_progress() -> dict:
        try:
            return json.loads((RUN_DIR / "progress.json").read_text())
        except (OSError, ValueError):
            return {}


    def watch_progress(stop: threading.Event, every: float = 60.0) -> None:
        last, t_start = -1, time.time()
        while not stop.wait(every):
            p = read_progress()
            if p.get("done") != last:
                last = p.get("done")
                print(f"[progress] {p.get('done')}/{p.get('total')} docs, "
                      f"{(time.time() - t_start) / 60:.0f} min this session", flush=True)


    done_before = read_progress().get("done", 0)
    print(f"=== {SHARD_RUN_ID}: {done_before}/{THIS_DOCS} docs already done; resuming the rest "
          f"(batch size {BATCH}, shard {SHARD}) ===")
    session = {"start": datetime.now(UTC).isoformat(), "end": None, "seconds": None,
               "docs_done_at_start": done_before, "exit_code": None}
    sessions.append(session)
    save_json(SESSIONS_PATH, sessions)
    run_cmd = [PY, "-m", "shipdoc", "predict", "run", "--config", f"configs/spike_{CONFIG}.yaml",
               "--run-id", RUN_ID, "--shard", SHARD, "--decision", str(DECISION_PATH),
               "--smoke-status", str(SMOKE_STATUS_PATH),
               "--expect-docs", str(EXPECTED_DOCS), "--expect-pages", str(EXPECTED_PAGES)]
    stop = threading.Event()
    threading.Thread(target=watch_progress, args=(stop,), daemon=True).start()
    t0 = time.time()
    try:
        rc, tail = run_stream(run_cmd)
    finally:
        stop.set()
        session.update(end=datetime.now(UTC).isoformat(), seconds=round(time.time() - t0, 1))
        save_json(SESSIONS_PATH, sessions)
    session["exit_code"] = rc
    save_json(SESSIONS_PATH, sessions)
    if rc != 0:
        oom = OOM_RE.search("\n".join(tail))
        raise RuntimeError(
            f"Test run stopped (exit {rc}{', CUDA out of memory' if oom else ''}) after "
            f"{read_progress().get('done')}/{THIS_DOCS} docs; last output: {tail[-3:]}. Everything "
            "done so far is on Drive: Run all again resumes. If it stops at the same point, "
            "report back."
        )
    print(f"--- test run finished (exit 0) in {(time.time() - t0) / 60:.1f} min this session ---")

else:
    print("skipped (zs test run): MODEL == 'zs' is false")


In [ ]:
if MODEL == "zs":
    # DETERMINISM (as 04): 5 seeded test documents are decoded AGAIN in a fresh process,
    # replaying the exact generate calls of the first pass; the serialized predictions must be
    # byte-identical. Counts only. A difference raises: nothing is assembled as submittable.
    if not SMOKE_PASSED or "BATCH" not in globals():
        raise RuntimeError("Smoke gate or batch decision missing: the determinism pass is NOT run.")
    det_cmd = [PY, "-m", "shipdoc", "predict", "determinism", "--config",
               f"configs/spike_{CONFIG}.yaml", "--run-id", RUN_ID, "--shard", SHARD,
               "--decision", str(DECISION_PATH)]
    t_det = time.time()
    rc, tail = run_stream(det_cmd)
    if rc != 0:
        raise RuntimeError(f"DETERMINISM CHECK FAILED (exit {rc}): {tail[-3:]}. Report back "
                           "(counts only).")
    sessions.append({"start": None, "end": None, "seconds": round(time.time() - t_det, 1),
                     "exit_code": 0, "stage": "determinism"})
    save_json(SESSIONS_PATH, sessions)

else:
    print("skipped (zs determinism pass): MODEL == 'zs' is false")


In [ ]:
if MODEL == "ft":
    # NATIVE: `shipdoc.predict_native verify` = the RESOLUTION CHECK (an adapter not trained at
    # the inference resolution is refused, fail closed), then the unchanged final-adapter
    # verification of 04c (one row per check). A failed check raises: NOTHING after it runs.
    CFG = f"configs/spike_{CONFIG}.yaml"
    verify_cmd = [PY, "-m", "shipdoc.predict_native", "verify", "--adapter-dir", str(ADAPTER),
                  "--pin", PINNED_SHA, "--config", CFG]
    rc, tail = run_stream(verify_cmd)
    if rc != 0:
        raise RuntimeError(
            f"ADAPTER REFUSED (exit {rc}): the inference is NOT started. Read the FAIL rows above "
            "(counts, hashes and SHAs only). GG: report back; do not edit the manifest to pass."
        )
    ADAPTER_MANIFEST = json.loads((ADAPTER / "manifest.json").read_text())
    TRAIN_SHA = ADAPTER_MANIFEST["code_sha"]
    if TRAIN_SHA != PINNED_SHA:
        print(f"training code {TRAIN_SHA[:7]} != pin {PINNED_SHA[:7]}: files that decide the "
              "model's input and output, changed between the two (empty = unchanged):")
        run_stream(["git", "diff", "--stat", TRAIN_SHA, PINNED_SHA, "--",
                    "src/shipdoc/train.py", "src/shipdoc/trainset.py", "src/shipdoc/prompts.py",
                    "src/shipdoc/extract.py", "configs/finetune_qwen35_4b_native.yaml", CFG])
    print(f"VERIFIED: final adapter, training precision {ADAPTER_MANIFEST.get('precision')}.")

else:
    print("skipped (adapter verification): MODEL == 'ft' is false")


In [ ]:
if MODEL == "ft":
    # INFER in ONE process (the merged model never leaves it), RESUMABLE per document. NATIVE:
    # the resolution gate again, then the unchanged 04c stages, printed as they happen: VERIFY
    # (again), MERGE (fp16 base + the adapter, 200 modules required), GUARD (12 bench pages at
    # batch 1 and at the chosen size on the MERGED model; any difference or an out-of-memory ->
    # batch 1, recorded; a resume keeps the stored decision), SMOKE (the gate of 04, 5 dev docs,
    # on the merged model), RUN (the 200 test documents). The gate is stamped into ft_run.json,
    # which `finalize` requires.
    import re
    import threading
    import time
    from datetime import UTC, datetime

    OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    SESSIONS_PATH = RUN_DIR / "sessions.json"  # wall-clock log, one entry per Run all
    sessions = json.loads(SESSIONS_PATH.read_text()) if SESSIONS_PATH.is_file() else []


    def save_json(path, obj) -> None:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(obj, indent=1))
        tmp.replace(path)  # atomic: a disconnect never leaves a half-written file


    def read_progress() -> dict:
        try:
            return json.loads((RUN_DIR / "progress.json").read_text())
        except (OSError, ValueError):
            return {}


    def watch_progress(stop: threading.Event, every: float = 60.0) -> None:
        last, t_start = -1, time.time()
        while not stop.wait(every):
            p = read_progress()
            if p.get("done") != last:
                last = p.get("done")
                print(f"[progress] {p.get('done')}/{p.get('total')} docs, "
                      f"{(time.time() - t_start) / 60:.0f} min this session", flush=True)


    done_before = read_progress().get("done", 0)
    print(f"=== {RUN_ID}: {done_before}/{EXPECTED_DOCS} docs already done; resuming the rest ===")
    session = {"start": datetime.now(UTC).isoformat(), "end": None, "seconds": None,
               "docs_done_at_start": done_before, "exit_code": None}
    sessions.append(session)
    save_json(SESSIONS_PATH, sessions)
    infer_cmd = [PY, "-m", "shipdoc.predict_native", "infer", "--adapter-dir", str(ADAPTER),
                 "--pin", PINNED_SHA, "--config", CFG, "--run-id", RUN_ID,
                 "--decision", str(DECISION_PATH), "--smoke-status", str(SMOKE_STATUS_PATH),
                 "--smoke-docs", SMOKE_DOCS, "--bench-docs", BENCH_DOCS,
                 "--batch-size", str(BATCH), "--expect-docs", str(EXPECTED_DOCS),
                 "--expect-pages", str(EXPECTED_PAGES)]
    stop = threading.Event()
    threading.Thread(target=watch_progress, args=(stop,), daemon=True).start()
    t0 = time.time()
    try:
        rc, tail = run_stream(infer_cmd)
    finally:
        stop.set()
        session.update(end=datetime.now(UTC).isoformat(), seconds=round(time.time() - t0, 1))
        save_json(SESSIONS_PATH, sessions)
    session["exit_code"] = rc
    save_json(SESSIONS_PATH, sessions)
    if rc != 0:
        oom = OOM_RE.search("\n".join(tail))
        raise RuntimeError(
            f"Inference stopped (exit {rc}{', CUDA out of memory' if oom else ''}) after "
            f"{read_progress().get('done')}/{EXPECTED_DOCS} docs; last output: {tail[-3:]}. "
            "Everything done so far is on Drive: Run all again resumes. If it stops at the same "
            "point, report back."
        )
    print(f"--- inference finished (exit 0) in {(time.time() - t0) / 60:.1f} min this session ---")

else:
    print("skipped (ft inference): MODEL == 'ft' is false")


In [ ]:
if MODEL == "ft":
    # DETERMINISM: 5 seeded test documents decoded AGAIN in a fresh process (new load, new
    # merge, new CUDA context), replaying the exact generate calls of the first pass;
    # byte-identical serialized predictions required. Behind the resolution gate. Counts only.
    det_cmd = [PY, "-m", "shipdoc.predict_native", "determinism", "--adapter-dir", str(ADAPTER),
               "--pin", PINNED_SHA, "--config", CFG, "--run-id", RUN_ID,
               "--decision", str(DECISION_PATH)]
    t_det = time.time()
    rc, tail = run_stream(det_cmd)
    if rc != 0:
        raise RuntimeError(f"DETERMINISM CHECK FAILED (exit {rc}): {tail[-3:]}. Report back "
                           "(counts only).")
    sessions.append({"start": None, "end": None, "seconds": round(time.time() - t_det, 1),
                     "exit_code": 0, "stage": "determinism"})
    save_json(SESSIONS_PATH, sessions)

else:
    print("skipped (ft determinism pass): MODEL == 'ft' is false")


In [ ]:
# ASSEMBLE + VALIDATION (CPU). `shipdoc predict assemble` runs the post-processing v1 (R1, R2, R3
# ON, the OCR cache passed, the frozen meta/slot_shapes.json) and every blocking check of 04: run
# complete, exactly the 200 test ids, 200 docs / 280 pages, JSON Schema, determinism, code SHA clean
# and equal to PINNED_SHA, model revision / config hash / seed 42 / prompt v2, real model and stack
# (zs also: the batch size equals the 02n run's). `shipdoc.predict_native finalize` then adds the
# 04b checks (rule switches all on, no R2 skipped, shapes sha256 = meta/slot_shapes.json, OCR cache
# complete, OCR re-check), for ft the 04c checks, and the native checks (config hash = native, no
# adapter on zs / the resolution gate stamped on ft + the validated v15 folder, no key outside
# doc_type / header / line_items). The manifest says `submission` = v15_<sha7> / v2n_<sha7>. A
# failure moves the predictions to test_predictions.REJECTED.json.
asm_cmd = [PY, "-m", "shipdoc", "predict", "assemble", "--config", CFG, "--run-id", RUN_ID,
           "--out-dir", str(SUBMISSION_DIR), "--schema", str(SCHEMA_PATH),
           "--expect-docs", str(EXPECTED_DOCS), "--expect-pages", str(EXPECTED_PAGES),
           "--expect-code-sha", PINNED_SHA, "--require-stack",
           "--smoke-status", str(SMOKE_STATUS_PATH), "--ocr-cache", str(OCR_ROOT)]
if SAMPLE_PATH.is_file():
    asm_cmd += ["--sample-submission", str(SAMPLE_PATH)]
if MODEL == "zs":
    asm_cmd += ["--dev-run-dir", str(DEV_RUN)]  # the batch contract against the 02n run
rc, tail = run_stream(asm_cmd)
OCR_RECHECK_PATH = META_DIR / "ocr_recheck.json"
fin_cmd = [PY, "-m", "shipdoc.predict_native", "finalize", "--model", MODEL, "--config", CFG,
           "--out-dir", str(SUBMISSION_DIR), "--run-dir", str(RUN_DIR),
           "--recheck", str(OCR_RECHECK_PATH), "--expect-pages", str(EXPECTED_PAGES)]
if MODEL == "ft":
    fin_cmd += ["--zs-test-dir", str(ZS_TEST)]
rc_fin, tail_fin = run_stream(fin_cmd)
if rc_fin != 0:
    raise RuntimeError(
        f"VALIDATION FAILED (assemble exit {rc}, finalize exit {rc_fin}): the files in "
        f"{SUBMISSION_DIR} are NOT submittable (test_predictions.REJECTED.json). Read "
        "validation_report.json there (counts and key paths only) and report back; do not submit."
    )


In [ ]:
# REVIEW FLAGS (CPU), a SEPARATE file: review_flags.json in the submission folder, never part of
# test_predictions.json. CALIBRATOR_FILE = None (the default) REFUSES here: refreeze the calibrator
# at native first. `shipdoc.predict_native flags` refuses a calibrator of the other arm, one that
# does not carry the native config hash (`run_config_hash`) and a submission at another hash; for
# ft it also needs the validated v15 folder (the agreement features). Then `shipdoc.flags` writes
# probabilities, accept / review flags and thresholds (no extracted value) and `check-flags` checks
# the file: exactly the 200 ids, no value string, thresholds and the calibrator sha256 recorded,
# no flag key inside the predictions. The flags are advisory: a failure here does not withhold the
# validated predictions.
if CALIBRATOR_FILE is None:
    raise RuntimeError(
        "CALIBRATOR_FILE is None: the review flags are NOT computed (the predictions in "
        f"{SUBMISSION_DIR} are VALIDATED and on Drive). Refreeze at native with "
        f"`scripts/freeze_calibrator.py --arm {MODEL}` after 02n and calibrate_v2 on the native "
        "run (zs) / the three native OOF runs and calibrate_v3 (ft), then set CALIBRATOR_FILE."
    )
CALIBRATOR = Path(CALIBRATOR_FILE)
if not CALIBRATOR.is_absolute():  # the clone first (committed artifact), then Drive
    in_clone = REPO / CALIBRATOR_FILE
    CALIBRATOR = in_clone if in_clone.is_file() else DRIVE_DIR / CALIBRATOR_FILE
if not CALIBRATOR.is_file():
    raise FileNotFoundError(f"CALIBRATOR_FILE {CALIBRATOR_FILE} is in neither the clone nor Drive")
if not (SUBMISSION_DIR / "test_predictions.json").is_file():
    raise RuntimeError("no validated test_predictions.json: the flags are not computed")
flags_cmd = [PY, "-m", "shipdoc.predict_native", "flags", "--model", MODEL, "--config", CFG,
             "--submission-dir", str(SUBMISSION_DIR), "--calibrator", str(CALIBRATOR),
             "--ocr-cache", str(OCR_ROOT), "--batch-size", str(FLAGS_ZS_BATCH),
             "--field-target", str(FIELD_TARGET), "--doc-target", str(DOC_TARGET),
             "--expect-docs", str(EXPECTED_DOCS)]
if MODEL == "ft":
    flags_cmd += ["--zs-test-dir", str(ZS_TEST)]
if REUSE_ACK_SPIKE_DIFF:
    flags_cmd.append("--ack-spike-diff")
rc, tail = run_stream(flags_cmd)
if rc != 0:
    raise RuntimeError(
        f"REVIEW FLAGS REFUSED (exit {rc}): {tail[-2:]}. The predictions in {SUBMISSION_DIR} are "
        "VALIDATED and on Drive; only the flags file is missing."
    )
check_cmd = [PY, "-m", "shipdoc.predict_native", "check-flags", "--config", CFG,
             "--out-dir", str(SUBMISSION_DIR), "--calibrator", str(CALIBRATOR),
             "--field-target", str(FIELD_TARGET), "--doc-target", str(DOC_TARGET),
             "--expect-docs", str(EXPECTED_DOCS)]
rc, tail = run_stream(check_cmd)
if rc != 0:
    print("!" * 78)
    print("REVIEW FLAGS CHECKS FAILED (exit", rc, "): see validation_report.json flags_checks.")
    print("The predictions are not affected; do not use review_flags.json.")
    print("!" * 78)


In [ ]:
# Completion banner. Reads only files on Drive: also runnable alone after a resume.
import json

bar = "=" * 78
BS = chr(92)
LOCAL_DEST = BS.join(["$SHIPDOC_SUBMISSIONS_DIR", SUB_NAME]) + BS
report_path = SUBMISSION_DIR / "validation_report.json"
manifest_path = SUBMISSION_DIR / "manifest.json"
report = json.loads(report_path.read_text()) if report_path.is_file() else None
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
print(bar)
if report is None:
    print(f"NOT ASSEMBLED: {report_path.name} missing. Run the cells above.")
    print(f"DONE {RUN_ID} assembled=False")
else:
    ok = bool(report["ok"])
    verdict = "VALIDATED: SUBMITTABLE" if ok else "REJECTED: DO NOT SUBMIT"
    kind = "zero-shot, base model" if MODEL == "zs" else "fine-tuned, final native adapter"
    print(f"TEST PREDICTIONS {SUB_PREFIX} ({kind}, NATIVE + R1-R3) " + verdict)
    print(bar)
    print(f"schema ok       : {report['schema_ok']}")
    print(f"docs found      : {report['docs_found']}")
    print(f"determinism ok  : {report['determinism_ok']}")
    for name, c in report["checks"].items():
        print(f"  {'PASS' if c['ok'] else 'FAIL'}  {name}: {c.get('detail', '')}")
    nat = manifest.get("native") or {}
    print(f"native config {nat.get('config')} hash {nat.get('config_hash')} max_pixels "
          f"{nat.get('max_pixels')}; model switch {nat.get('model')}")
    if MODEL == "ft":
        ft = manifest.get("ft") or {}
        print(f"final adapter {str(ft.get('adapter_sha256'))[:12]}  training code "
              f"{str(ft.get('train_code_sha'))[:7]}  pin {PINNED_SHA[:7]}  precision "
              f"{ft.get('train_precision')}")
        mg, gd = ft.get("merge") or {}, ft.get("guard") or {}
        print(f"merge: {mg.get('n_lora_modules_merged')} modules, {mg.get('merge_dtype')}, "
              f"{mg.get('load_and_merge_s')} s; guard ran={gd.get('ran')} ok={gd.get('ok')} "
              f"fallback_to_1={gd.get('fallback_to_1')}")
        zt = nat.get("zs_test") or {}
        print(f"zero-shot traces: {zt.get('dir')} (batch {zt.get('batch_size')}, code "
              f"{str(zt.get('code_sha'))[:7]})")
    print(f"code {str(manifest.get('code_sha'))[:12]}  model {manifest.get('model', {}).get('id')} "
          f"@ {str(manifest.get('model', {}).get('revision'))[:12]}  config "
          f"{manifest.get('config', {}).get('hash')}")
    print(f"batch size {manifest.get('batch_size')} ({manifest.get('batch_size_source')}); "
          f"contract: {manifest.get('batch_size_contract')}; "
          f"smoke {(manifest.get('smoke') or {}).get('state')}")
    ocr_t = (manifest.get("ocr") or {}).get("timing") or {}
    print(f"OCR: {ocr_t.get('pages')} pages, {ocr_t.get('seconds_total')} s total, "
          f"{ocr_t.get('seconds_per_page_mean')} s/page mean, p95 "
          f"{ocr_t.get('seconds_per_page_p95')}, device {ocr_t.get('device')}")
    post = manifest.get("post_rules") or {}
    print(f"rules R1/R2/R3: touched docs {post.get('touched_docs')}, skipped {post.get('skipped')}")
    tm = manifest.get("timings", {})
    print(f"wall-clock {tm.get('wall_clock_s')} s over {tm.get('sessions')} session(s); model time "
          f"{tm.get('model_time_s')} s; smoke {tm.get('smoke_s')} s")
    rf = manifest.get("review_flags")
    flags_ok = bool(report.get("flags_ok")) if "flags_ok" in report else None
    if rf is None:
        print("REVIEW FLAGS: not computed (the flags cell did not run or failed)")
    else:
        s = rf.get("summary") or {}
        state = "OK" if flags_ok else "CHECKS FAILED"
        print(f"REVIEW FLAGS {state} (arm {rf.get('arm')}, calibrator "
              f"sha256 {str(rf.get('calibrator_sha256'))[:12]}): {s.get('n_docs')} docs, "
              f"auto-accept {s.get('n_docs_auto_accept')}, fields accept {s.get('n_accept')} / "
              f"review {s.get('n_review')}; field tau {rf['thresholds'].get('field_tau')}, doc tau "
              f"{rf['thresholds'].get('doc_tau')}")
        for name, c in (report.get("flags_checks") or {}).items():
            print(f"  {'PASS' if c['ok'] else 'FAIL'}  {name}: {c.get('detail', '')}")
    print(f"Drive folder: {SUBMISSION_DIR}")
    print("GG: download these files from that Drive folder:")
    wanted = ["test_predictions.json" if ok else "test_predictions.REJECTED.json",
              "review_flags.json", "trace.jsonl", "manifest.json", "validation_report.json",
              "rules.jsonl", "ocr_timing.json"]
    for name in wanted:
        path = SUBMISSION_DIR / name
        size = f"{path.stat().st_size / 1e6:8.2f} MB" if path.is_file() else "MISSING"
        print(f"  {name:<34}{size}")
    print(f"into the LOCAL folder  {LOCAL_DEST}")
    print("(outside the repo; submissions/ is gitignored). Never paste document values into chat.")
    print(f"DONE {RUN_ID} assembled=True submittable={ok} flags_ok={flags_ok}")
print(bar)
